# 🫀 Left Atrium Segmentation với DINOv2
**Notebook này train mô hình trên GPU của Google Colab.**

### Các bước:
1. Mount Google Drive
2. Cài thư viện
3. Upload / kiểm tra dữ liệu
4. Chạy training
5. Đánh giá kết quả

> **Lưu ý:** Vào `Runtime → Change runtime type → T4 GPU` trước khi chạy.

## 0. Kiểm tra GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: Khong co GPU! Vao Runtime -> Change runtime type -> T4 GPU')

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# === CAU HINH DUONG DAN ===
# Thay doi PROJECT_PATH cho phu hop voi vi tri tren Drive cua ban
PROJECT_PATH = '/content/drive/MyDrive/Left_atrium_segmentation'

import os, sys
os.chdir(PROJECT_PATH)
sys.path.insert(0, PROJECT_PATH)
print('Working dir:', os.getcwd())
print('Files:', os.listdir('.'))

## 2. Cài thư viện

In [ ]:
!pip install -q nibabel scikit-learn tqdm
print('Done!')

## 3. Kiểm tra dữ liệu

**Lựa chọn A (Khuyên dùng):** Upload thư mục `data/precomputed/` từ máy tính lên Drive  
→ Bỏ qua bước 3a, chạy thẳng bước 4

**Lựa chọn B:** Upload thư mục `data/raw_3d/` và chạy lại từ đầu

In [ ]:
# Kiem tra xem precomputed features da co chua
precomp_path = 'data/precomputed'
if os.path.exists(os.path.join(precomp_path, 'train', 'features')):
    n_train = len(os.listdir(os.path.join(precomp_path, 'train', 'features')))
    n_val   = len(os.listdir(os.path.join(precomp_path, 'val',   'features')))
    n_test  = len(os.listdir(os.path.join(precomp_path, 'test',  'features')))
    print(f'[OK] Pre-computed features da san sang!')
    print(f'  Train: {n_train} | Val: {n_val} | Test: {n_test}')
    USE_PRECOMPUTED = True
else:
    print('[!] Chua co pre-computed features. Can chay lai tu dau.')
    USE_PRECOMPUTED = False

## 3a. (Tùy chọn) Chạy lại từ đầu nếu chưa có precomputed features
**Bỏ qua cell này nếu đã có `data/precomputed/`**

In [ ]:
if not USE_PRECOMPUTED:
    print('=== BUOC 1: Chuan bi du lieu 2D ===')
    !python src/prepare_data.py

    print('\n=== BUOC 2: Trich xuat DINOv2 features (chay 1 lan) ===')
    !python src/extract_features.py --model vit_small --batch_size 16

    print('\nHoan tat! Da san sang de train.')

## 4. 🚀 Huấn luyện mô hình (NHANH trên GPU!)

In [ ]:
# Kiem tra GPU mot lan nua truoc khi train
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training tren: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# === CAU HINH TRAINING ===
MODEL      = 'vit_small'   # vit_small | vit_base | vit_large
BATCH_SIZE = 64            # GPU T4: 32-64, A100: 128
EPOCHS     = 35
LR         = 1e-3
PATIENCE   = 7

!python src/train_fast.py \
    --model {MODEL} \
    --batch_size {BATCH_SIZE} \
    --epochs {EPOCHS} \
    --lr {LR} \
    --patience {PATIENCE} \
    --num_workers 2 \
    --save_dir results

## 5. 📊 Đánh giá kết quả

In [ ]:
!python src/evaluate.py \
    --model vit_small \
    --checkpoint results/best_decoder.pth \
    --batch_size 32 \
    --num_workers 2 \
    --max_vis 20

## 6. 📈 Xem biểu đồ training

In [ ]:
import json
import matplotlib.pyplot as plt

with open('results/training_history.json') as f:
    history = json.load(f)

epochs = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric, title in zip(axes,
    ['loss', 'dice', 'iou'],
    ['Loss', 'Dice Score', 'IoU (Jaccard)']):
    ax.plot(epochs, history[f'train_{metric}'], 'b-o', markersize=4, label='Train')
    ax.plot(epochs, history[f'val_{metric}'],   'r-o', markersize=4, label='Val')
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Training History - DINOv2 Left Atrium Segmentation', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('results/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Bieu do da luu tai results/training_curves.png')

## 7. Xem ảnh trực quan hoá

In [ ]:
import glob
from IPython.display import Image, display

vis_files = sorted(glob.glob('results/visualizations/*.png'))[:5]
for f in vis_files:
    print(os.path.basename(f))
    display(Image(f, width=900))